# AI Travel Operations Agent

## Project Introduction

The AI Travel Operations Agent is an intelligent multi-agent system designed to automate and simplify travel planning.

The system uses Large Language Models (LLMs), LangGraph, and the Model Context Protocol (MCP) to coordinate specialized AI agents.

Each agent performs a specific task, such as transportation research, hotel discovery, weather analysis, activity recommendations, budget estimation, and itinerary verification.

### Project Objectives

1. Understand user travel requirements.
2. Discover suitable transportation options.
3. Find hotels based on budget and preferences.
4. Analyze weather conditions.
5. Recommend tourist attractions and activities.
6. Estimate total travel expenses.
7. Generate a personalized travel itinerary.
8. Verify information using connected data sources.

### Example User Request

"Plan a 3-day trip from Hyderabad to Goa for 2 travelers with a total budget of INR 15,000."

### Expected Output

- Transportation recommendations
- Hotel recommendations
- Weather information
- Tourist attractions
- Day-wise travel itinerary
- Estimated total budget

## Technology Stack

| Technology | Purpose |
|---|---|
| Python | Core development language |
| Jupyter Notebook | Development and experimentation |
| Groq API | LLM inference |
| LangChain | LLM integration |
| LangGraph | Multi-agent workflow orchestration |
| MCP | Standardized tool integration |
| Pydantic | Data validation |
| HTTPX | External API requests |
| FastAPI | Backend API development |
| PostgreSQL | Persistent data storage |
| React | Frontend development |
| Tailwind CSS | User interface styling |

## System Architecture

User Travel Request
        |
        v
Travel Request Validation
        |
        v
LangGraph Orchestrator
        |
        +---- Transport Agent
        |
        +---- Hotel Agent
        |
        +---- Weather Agent
        |
        +---- Activities Agent
        |
        +---- Budget Agent
        |
        +---- Review Agent
        |
        v
External APIs and MCP Tools
        |
        v
Final Travel Itinerary

# Import Required Libraries

In [1]:
import os
import sys
import platform

from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [2]:
print("Project: AI Travel Operations Agent")
print("Python Version:", sys.version.split()[0])
print("Operating System:", platform.system())
print("Working Directory:", Path.cwd())

Project: AI Travel Operations Agent
Python Version: 3.12.4
Operating System: Windows
Working Directory: C:\Users\Rohit singh\Desktop\ai-travel-operations-agent\notebooks


## Environment Configuration

Environment variables are used to store sensitive configuration values, such as API keys.

The GROQ_API_KEY is stored in a .env file instead of being hardcoded in the source code.

In [3]:
PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing.")

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [4]:
MODEL_NAME = "openai/gpt-oss-120b"

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq client initialized successfully.")
print("Selected Model:", MODEL_NAME)

Groq client initialized successfully.
Selected Model: openai/gpt-oss-120b


# Test LLM Response

In [5]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an intelligent AI Travel Assistant. "
                "Provide clear, practical, and structured travel guidance."
            )
        },
        {
            "role": "user",
            "content": (
                "Introduce yourself and explain how you can "
                "assist users with travel planning."
            )
        }
    ],
    temperature=0.3,
    max_completion_tokens=300
)

answer = response.choices[0].message.content

print("AI Travel Assistant Response:\n")
print(answer)

AI Travel Assistant Response:

**Hello! I’m your AI Travel Assistant – a friendly, knowledgeable companion who helps you turn travel ideas into smooth, well‑organized trips.**  

### What I can do for you
| Area | How I help | Example output |
|------|------------|----------------|
| **Trip brainstorming** | Suggest destinations, themes, and activities based on interests, budget, season, and travel style. | “A 10‑day cultural road‑trip through Portugal’s historic towns and coastal villages.” |
| **Itinerary building** | Create day‑by‑day schedules, balance sightseeing with downtime, and include opening hours, travel times, and optional extras. | Detailed 7‑day itinerary for Kyoto with morning temple visits, afternoon tea houses, and evening geisha‑district walks. |
| **Flights & transportation** | Find optimal flight routes, compare airlines, suggest layovers, and recommend ground‑transport options (trains, buses, car rentals, rideshares). | “Best round‑trip flights from NYC to Bangkok

# Create a Reusable LLM Function

In [6]:
def generate_llm_response(
    user_prompt: str,
    system_prompt: str = "You are a helpful AI Travel Assistant."
) -> str:

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.3,
        max_completion_tokens=500
    )

    return response.choices[0].message.content or ""

In [7]:
result = generate_llm_response(
    user_prompt="Suggest 5 popular tourist attractions in Goa."
)

print(result)

Here are five of the most popular tourist attractions in Goa, each offering a different flavor of the state’s vibrant culture, natural beauty, and history:

| # | Attraction | Why It’s a Must‑Visit | Highlights & Tips |
|---|------------|----------------------|-------------------|
| **1** | **Baga & Calangute Beaches** | The iconic “Party Beaches” of North Goa. | • Golden sand, water‑sports (parasailing, jet‑ski, banana boat). <br>• Lively beach shacks serving fresh seafood and Goan feni. <br>• Best visited in the late afternoon for sunset and the bustling night market at Calangute. |
| **2** | **Fort Aguada & Lighthouse** | A well‑preserved 17th‑century Portuguese fort with panoramic sea views. | • Walk the ramparts for sweeping vistas of the Arabian Sea. <br>• Climb the lighthouse (still operational) for a photo‑worthy panorama. <br>• Nearby Sinquerim Beach is quieter for a post‑fort dip. |
| **3** | **Basilica of Bom Bom (Our Lady of the Immaculate Conception)** | Goa’s most famous 

# Create the Travel Request Schema

In [8]:
# Import Pydantic
from pydantic import BaseModel, Field, model_validator
from datetime import date
from typing import Optional

In [9]:
# Define the Schema

class TravelRequest(BaseModel):
    source: str = Field(
        ...,
        min_length=2,
        description="Starting city of the trip"
    )

    destination: str = Field(
        ...,
        min_length=2,
        description="Destination city"
    )

    start_date: date = Field(
        ...,
        description="Trip start date"
    )

    end_date: date = Field(
        ...,
        description="Trip end date"
    )

    travelers: int = Field(
        default=1,
        ge=1,
        le=20,
        description="Number of travelers"
    )

    budget: float = Field(
        ...,
        gt=0,
        description="Total trip budget in INR"
    )

    currency: str = "INR"

    preferences: list[str] = Field(
        default_factory=list,
        description="User travel preferences"
    )

    @model_validator(mode="after")
    def validate_trip(self):
        if self.end_date < self.start_date:
            raise ValueError(
                "End date must be on or after start date."
            )

        if self.source.strip().casefold() == self.destination.strip().casefold():
            raise ValueError(
                "Source and destination must be different."
            )

        return self

In [10]:
# Test the Schema

travel_request = TravelRequest(
    source="Hyderabad",
    destination="Goa",
    start_date="2026-11-10",
    end_date="2026-11-12",
    travelers=2,
    budget=15000,
    preferences=["Beaches", "Local Food", "Sightseeing"]
)

print(travel_request)

source='Hyderabad' destination='Goa' start_date=datetime.date(2026, 11, 10) end_date=datetime.date(2026, 11, 12) travelers=2 budget=15000.0 currency='INR' preferences=['Beaches', 'Local Food', 'Sightseeing']


In [11]:
travel_data = travel_request.model_dump(mode="json")

print(travel_data)

{'source': 'Hyderabad', 'destination': 'Goa', 'start_date': '2026-11-10', 'end_date': '2026-11-12', 'travelers': 2, 'budget': 15000.0, 'currency': 'INR', 'preferences': ['Beaches', 'Local Food', 'Sightseeing']}


In [12]:
# Create the Shared Agent State

from typing import TypedDict, Optional, Any

class TravelAgentState(TypedDict, total=False):
    travel_request: dict[str, Any]

    transport_options: list[dict[str, Any]]
    hotel_options: list[dict[str, Any]]
    weather_info: dict[str, Any]
    activities: list[dict[str, Any]]

    budget_summary: dict[str, Any]
    review_result: dict[str, Any]

    itinerary: dict[str, Any]
    errors: list[str]

In [13]:
# Initialize the Shared State

initial_state: TravelAgentState = {
    "travel_request": travel_data,
    "transport_options": [],
    "hotel_options": [],
    "weather_info": {},
    "activities": [],
    "budget_summary": {},
    "review_result": {},
    "itinerary": {},
    "errors": []
}

print("Initial Travel Agent State:")
print(initial_state)

Initial Travel Agent State:
{'travel_request': {'source': 'Hyderabad', 'destination': 'Goa', 'start_date': '2026-11-10', 'end_date': '2026-11-12', 'travelers': 2, 'budget': 15000.0, 'currency': 'INR', 'preferences': ['Beaches', 'Local Food', 'Sightseeing']}, 'transport_options': [], 'hotel_options': [], 'weather_info': {}, 'activities': [], 'budget_summary': {}, 'review_result': {}, 'itinerary': {}, 'errors': []}


In [14]:
# Validate Incorrect Input
# Test Validation

try:
    invalid_request = TravelRequest(
        source="Hyderabad",
        destination="Goa",
        start_date="2026-11-12",
        end_date="2026-11-10",
        travelers=2,
        budget=15000
    )
except Exception as error:
    print("Validation Error:")
    print(error)

Validation Error:
1 validation error for TravelRequest
  Value error, End date must be on or after start date. [type=value_error, input_value={'source': 'Hyderabad', '...rs': 2, 'budget': 15000}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


# Groq LLM Integration & Reusable LLM Service

In [15]:
# Define ask_llm()

def ask_llm(
    prompt: str,
    system_prompt: str = "You are a helpful AI Travel Assistant.",
    temperature: float = 0.3,
    max_tokens: int = 500
) -> str:
    """
    Send a prompt to the Groq LLM and return its response.
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature,
        max_completion_tokens=max_tokens
    )

    return response.choices[0].message.content or ""

In [16]:
# est the LLM Service

result = ask_llm(
    prompt="Suggest five popular tourist attractions in Goa.",
    system_prompt=(
        "You are a travel planning assistant. "
        "Give clear and concise recommendations."
    )
)

print(result)

Here are five must‑see attractions in Goa:

| # | Attraction | Why Visit? |
|---|------------|------------|
| 1 | **Baga Beach** (North Goa) | Golden sand, lively shacks, water sports, and vibrant nightlife. |
| 2 | **Basilica of Bom Bom (Our Lady of the Immaculate Conception)** | Iconic 16th‑century church with stunning Baroque architecture and a famous statue of the Virgin Mary. |
| 3 | **Fort Aguada & Lighthouse** (Sinquerim) | Well‑preserved Portuguese fort offering panoramic sea views and a historic lighthouse. |
| 4 | **Dudhsagar Waterfalls** (South Goa) | Majestic tiered falls set in a lush jungle; best seen by jeep safari or train ride. |
| 5 | **Old Goa (UNESCO World Heritage Site)** | Cluster of historic churches and cathedrals—Basilica of St Francis of Assisi, Se Cathedral, etc.—showcasing Goa’s colonial heritage. |

These spots give you a mix of beach fun, cultural history, and natural beauty—perfect for a well‑rounded Goa itinerary. Enjoy your trip!


In [17]:
# Generate a Travel Summary

import json

travel_prompt = f"""
Analyze the following travel request.

Travel request:
{json.dumps(travel_data, indent=2)}

Provide:
1. A short summary of the trip.
2. The main travel preferences.
3. Important planning considerations.
4. Information that still needs verification.
"""

travel_summary = ask_llm(
    prompt=travel_prompt,
    system_prompt=(
        "You are an AI travel planning analyst. "
        "Do not invent live prices or availability."
    )
)

print(travel_summary)

**1. Short Summary of the Trip**  
- **Origin & Destination:** Hyderabad → Goa  
- **Dates:** 10 Nov 2026 (arrival) – 12 Nov 2026 (departure) – a 2‑night, 3‑day getaway.  
- **Travelers:** 2 adults.  
- **Budget:** ₹15,000 total (≈ ₹7,500 per person).  
- **Key Interests:** Beaches, local Goan cuisine, and sightseeing.  

**2. Main Travel Preferences**  
| Preference | Typical Activities / Highlights |
|------------|---------------------------------|
| **Beaches** | Sun‑sand‑sea at Calangute, Baga, Anjuna, Palolem; water sports (optional) |
| **Local Food** | Goan seafood (fish curry, prawn balchão, xacuti), pork vindaloo, feni tasting, beach shacks, market stalls |
| **Sightseeing** | Old Goa churches (Basilica of Bom Jesus, Se Cathedral), Fort Aguada, Chapora Fort, spice farms, Dolphin‑watching tours (if time permits) |

**3. Important Planning Considerations**  

| Area | What to Keep in Mind |
|------|----------------------|
| **Transportation to Goa** | • **Air:** Hyderabad → Goa 

In [18]:
# Add Basic Error Handling

import logging
from groq import APIConnectionError, APIStatusError, RateLimitError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("travel_agent")


def safe_ask_llm(
    prompt: str,
    system_prompt: str = "You are a helpful AI Travel Assistant."
) -> str:
    try:
        return ask_llm(
            prompt=prompt,
            system_prompt=system_prompt
        )

    except RateLimitError:
        logger.warning("Groq API rate limit reached.")
        return "The AI service is temporarily rate-limited."

    except APIConnectionError:
        logger.exception("Unable to connect to Groq.")
        return "The AI service is currently unreachable."

    except APIStatusError as error:
        logger.error("Groq API returned status %s.", error.status_code)
        return "The AI service returned an API error."

In [19]:
# Test Error-Handled Function

result = safe_ask_llm(
    "Explain why travel budgets should include emergency expenses."
)

print(result)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Why a Travel Budget Needs an Emergency‑Expense Cushion**

| Reason | What It Covers | Why It Matters |
|--------|----------------|----------------|
| **Medical emergencies** | Hospital visits, doctor’s fees, prescription meds, travel‑insurance deductibles | Health care costs can sky‑rocket abroad, especially in countries without universal coverage. Even a minor injury can become a costly surprise. |
| **Trip disruptions** | Flight cancellations or delays, missed connections, last‑minute re‑booking, extra accommodation | Weather, strikes, technical glitches, or political events can force you to change plans on short notice. Re‑booking fees and overnight stays add up quickly. |
| **Lost or stolen belongings** | Replacement of passports, visas, credit cards, phones, cameras, clothing | A stolen bag or a misplaced passport can halt a trip entirely until you obtain replacements—often at premium prices for expedited services. |
| **Natural disasters or civil unrest** | Evacuation costs, em

# Transport Agent

A Transport Agent is a specialized AI component responsible for analyzing transportation requirements and comparing available travel options based on cost, duration, convenience, and user preferences.

In [20]:
# Create Sample Transport Data

transport_data = [
    {
        "mode": "Flight",
        "price_per_person": 4500,
        "duration_hours": 1.5,
        "comfort": 5
    },
    {
        "mode": "Train",
        "price_per_person": 1200,
        "duration_hours": 14,
        "comfort": 4
    },
    {
        "mode": "Bus",
        "price_per_person": 1000,
        "duration_hours": 16,
        "comfort": 3
    }
]

print("Sample transport data created successfully!")

Sample transport data created successfully!


In [21]:
# Calculate Total Transport Cost

def calculate_transport_cost(
    price_per_person: float,
    travelers: int
) -> float:
    return price_per_person * travelers


travelers = travel_data["travelers"]

for option in transport_data:
    option["total_cost"] = calculate_transport_cost(
        option["price_per_person"],
        travelers
    )

transport_data

[{'mode': 'Flight',
  'price_per_person': 4500,
  'duration_hours': 1.5,
  'comfort': 5,
  'total_cost': 9000},
 {'mode': 'Train',
  'price_per_person': 1200,
  'duration_hours': 14,
  'comfort': 4,
  'total_cost': 2400},
 {'mode': 'Bus',
  'price_per_person': 1000,
  'duration_hours': 16,
  'comfort': 3,
  'total_cost': 2000}]

In [22]:
# Rank Transport Options
# Define Ranking Logic

def rank_transport_options(
    options: list[dict],
    budget: float
) -> list[dict]:
    ranked_options = []

    for option in options:
        item = option.copy()

        item["within_budget"] = item["total_cost"] <= budget

        ranked_options.append(item)

    ranked_options.sort(
        key=lambda item: (
            not item["within_budget"],
            item["total_cost"],
            item["duration_hours"]
        )
    )

    return ranked_options

In [23]:
# Run Ranking

budget = travel_data["budget"]

ranked_transport = rank_transport_options(
    transport_data,
    budget
)

for index, option in enumerate(ranked_transport, start=1):
    print(f"\nRank: {index}")
    print("Mode:", option["mode"])
    print("Total Cost:", option["total_cost"])
    print("Duration (hours):", option["duration_hours"])
    print("Within Budget:", option["within_budget"])


Rank: 1
Mode: Bus
Total Cost: 2000
Duration (hours): 16
Within Budget: True

Rank: 2
Mode: Train
Total Cost: 2400
Duration (hours): 14
Within Budget: True

Rank: 3
Mode: Flight
Total Cost: 9000
Duration (hours): 1.5
Within Budget: True


In [24]:
# Use the LLM to Explain Options
# Generate a Transport Recommendation

import json

transport_prompt = f"""
You are a transport planning assistant.

Travel request:
{json.dumps(travel_data, indent=2)}

Transport options:
{json.dumps(ranked_transport, indent=2)}

Compare the options based on total cost, duration, and comfort.

Recommend a suitable option and explain the trade-offs.
Clearly mention that these are sample estimates, not live prices.
Do not invent schedules or availability.
"""

transport_recommendation = ask_llm(
    prompt=transport_prompt,
    system_prompt=(
        "You are a careful travel analyst. "
        "Use only the supplied data for numerical comparisons."
    )
)

print(transport_recommendation)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


**Comparison of the three sample transport options (Hyderabad → Goa, 2 travelers)**  

| Mode   | Total cost (INR) | Travel duration (hours) | Comfort rating* |
|--------|------------------|--------------------------|-----------------|
| Bus    | 2,000            | 16                       | 3 |
| Train  | 2,400            | 14                       | 4 |
| Flight | 9,000            | 1.5                      | 5 |

\*Comfort is a simple 1‑5 scale provided in the data (5 = most comfortable).

### Trade‑off summary  

| Factor | Bus | Train | Flight |
|--------|-----|-------|--------|
| **Cost** – cheapest option, leaves most of the INR 15,000 budget for accommodation, food, and activities. | Moderate increase over bus (≈ 20 % more). Still well under the budget. | Highest cost (≈ 4.5 ×  the bus price). Consumes a larger share of the budget, but still leaves ≈ 6,000 INR for other expenses. |
| **Duration** – longest journey; you’ll spend most of a day traveling, which reduces time for si

In [25]:
# Save Results in Agent State
# Update Shared State

initial_state["transport_options"] = ranked_transport

print("Transport Agent results saved in shared state.")
initial_state["transport_options"]

Transport Agent results saved in shared state.


[{'mode': 'Bus',
  'price_per_person': 1000,
  'duration_hours': 16,
  'comfort': 3,
  'total_cost': 2000,
  'within_budget': True},
 {'mode': 'Train',
  'price_per_person': 1200,
  'duration_hours': 14,
  'comfort': 4,
  'total_cost': 2400,
  'within_budget': True},
 {'mode': 'Flight',
  'price_per_person': 4500,
  'duration_hours': 1.5,
  'comfort': 5,
  'total_cost': 9000,
  'within_budget': True}]

# Prepare the travel request and sample data

In [26]:
from datetime import date
import json

# User travel request
travel_request = {
    "source": "Hyderabad",
    "destination": "Goa",
    "start_date": "2026-11-10",
    "end_date": "2026-11-12",
    "travelers": 2,
    "budget": 15000,
    "currency": "INR",
    "preferences": ["Beaches", "Local Food", "Sightseeing"]
}

# Demo transport options (not live prices)
transport_data = [
    {"mode": "Flight", "price_per_person": 4500, "duration_hours": 1.5, "comfort": 5},
    {"mode": "Train", "price_per_person": 1200, "duration_hours": 14, "comfort": 4},
    {"mode": "Bus", "price_per_person": 1000, "duration_hours": 16, "comfort": 3}
]

# Demo hotel options (not live prices)
hotel_data = [
    {"name": "Budget Guesthouse", "price_per_night": 1200, "rating": 3.8},
    {"name": "Beachside Hotel", "price_per_night": 2500, "rating": 4.3},
    {"name": "Premium Resort", "price_per_night": 5000, "rating": 4.7}
]

# Placeholder weather data; replace with a weather API later
weather_data = {
    "source": "Demo data - not a live forecast",
    "summary": "Check the forecast closer to the travel dates."
}

print("Travel request and sample data are ready.")

Travel request and sample data are ready.


# Define all six agents

In [27]:
def transport_agent(request, options):
    travelers = request["travelers"]
    budget = request["budget"]

    results = []

    for option in options:
        item = option.copy()
        item["total_cost"] = item["price_per_person"] * travelers
        item["within_budget"] = item["total_cost"] <= budget
        results.append(item)

    return sorted(
        results,
        key=lambda x: (
            not x["within_budget"],
            x["total_cost"],
            x["duration_hours"]
        )
    )


def hotel_agent(request, options):
    travelers = request["travelers"]
    start = date.fromisoformat(request["start_date"])
    end = date.fromisoformat(request["end_date"])
    nights = (end - start).days

    if nights < 1:
        raise ValueError("End date must be after start date.")

    results = []

    for option in options:
        item = option.copy()
        item["nights"] = nights
        item["total_cost"] = item["price_per_night"] * nights
        item["total_cost_for_group"] = item["total_cost"]
        item["cost_per_person"] = item["total_cost"] / travelers
        results.append(item)

    return sorted(results, key=lambda x: x["total_cost"])


def weather_agent(request, weather):
    return {
        "destination": request["destination"],
        "travel_dates": [
            request["start_date"],
            request["end_date"]
        ],
        "weather_summary": weather["summary"],
        "source": weather["source"],
        "recommendation": (
            "Verify the actual forecast before departure."
        )
    }


def activities_agent(request):
    prompt = f"""
    Suggest 5 activities for this trip.

    Travel request:
    {json.dumps(request, indent=2)}

    Return a JSON array of objects with these fields:
    name, description, estimated_cost_per_person.

    Use reasonable illustrative estimates in INR.
    Do not claim live availability or current prices.
    Return only valid JSON.
    """

    raw_response = ask_llm(
        prompt,
        system_prompt=(
            "You are a travel activities planner. "
            "Clearly treat costs as estimates."
        )
    )

    try:
        activities = json.loads(raw_response)

        if not isinstance(activities, list):
            raise ValueError("Expected a JSON list.")

        for activity in activities:
            if not isinstance(activity, dict):
                raise ValueError("Invalid activity format.")

            if not isinstance(activity.get("name"), str):
                raise ValueError("Activity name is missing.")

            cost = float(activity.get("estimated_cost_per_person", 0))

            if cost < 0:
                raise ValueError("Activity cost cannot be negative.")

            activity["estimated_cost_per_person"] = cost

        return activities

    except (json.JSONDecodeError, TypeError, ValueError):
        # Safe fallback if the model returns invalid JSON
        return [
            {
                "name": "Beach visit",
                "description": "Enjoy a beach visit and sightseeing.",
                "estimated_cost_per_person": 300
            },
            {
                "name": "Local food experience",
                "description": "Explore local cuisine.",
                "estimated_cost_per_person": 500
            }
        ]


def budget_agent(request, transport, hotels, activities):
    travelers = request["travelers"]
    total_budget = request["budget"]

    # Use the cheapest sample transport option
    selected_transport = transport[0]

    # Use the cheapest sample hotel option
    selected_hotel = hotels[0]

    transport_cost = selected_transport["total_cost"]
    hotel_cost = selected_hotel["total_cost"]

    # Estimate activity costs for all travelers
    activities_cost = sum(
        item["estimated_cost_per_person"] * travelers
        for item in activities
    )

    # Illustrative food and local transport allowance
    food_cost = 1500 * travelers
    local_transport_cost = 500 * travelers

    estimated_total = (
        transport_cost
        + hotel_cost
        + activities_cost
        + food_cost
        + local_transport_cost
    )

    return {
        "currency": request["currency"],
        "total_budget": total_budget,
        "transport_cost": transport_cost,
        "hotel_cost": hotel_cost,
        "activities_cost": activities_cost,
        "food_estimate": food_cost,
        "local_transport_estimate": local_transport_cost,
        "estimated_total": estimated_total,
        "remaining_budget": total_budget - estimated_total,
        "within_budget": estimated_total <= total_budget,
        "note": "All costs are illustrative estimates."
    }


def review_agent(request, budget, transport, hotels, activities):
    issues = []

    if not transport:
        issues.append("No transport options available.")

    if not hotels:
        issues.append("No hotel options available.")

    if not activities:
        issues.append("No activities were generated.")

    if budget["estimated_total"] > request["budget"]:
        issues.append("Estimated trip cost exceeds the budget.")

    if budget["estimated_total"] < 0:
        issues.append("Estimated total cost is invalid.")

    return {
        "status": "PASS" if not issues else "NEEDS_REVIEW",
        "issues": issues,
        "verified_live_prices": False,
        "message": (
            "Plan passed basic checks."
            if not issues
            else "Review the issues before using this plan."
        )
    }


print("All six agent functions defined successfully.")

All six agent functions defined successfully.


# Run the complete agent workflow

In [28]:
# Initialize shared state
state = {
    "travel_request": travel_request,
    "transport_options": [],
    "hotel_options": [],
    "weather_info": {},
    "activities": [],
    "budget_summary": {},
    "review_result": {},
    "itinerary": {},
    "errors": []
}

try:
    # Agent 1: Transport
    state["transport_options"] = transport_agent(
        travel_request, transport_data
    )

    # Agent 2: Hotels
    state["hotel_options"] = hotel_agent(
        travel_request, hotel_data
    )

    # Agent 3: Weather
    state["weather_info"] = weather_agent(
        travel_request, weather_data
    )

    # Agent 4: Activities (uses Groq LLM)
    state["activities"] = activities_agent(travel_request)

    # Agent 5: Budget
    state["budget_summary"] = budget_agent(
        travel_request,
        state["transport_options"],
        state["hotel_options"],
        state["activities"]
    )

    # Agent 6: Review
    state["review_result"] = review_agent(
        travel_request,
        state["budget_summary"],
        state["transport_options"],
        state["hotel_options"],
        state["activities"]
    )

    # Create the final itinerary
    state["itinerary"] = {
        "source": travel_request["source"],
        "destination": travel_request["destination"],
        "start_date": travel_request["start_date"],
        "end_date": travel_request["end_date"],
        "travelers": travel_request["travelers"],
        "selected_transport": state["transport_options"][0],
        "selected_hotel": state["hotel_options"][0],
        "activities": state["activities"],
        "weather": state["weather_info"],
        "budget": state["budget_summary"],
        "review": state["review_result"]
    }

    print("All six agents executed successfully!")

except Exception as error:
    state["errors"].append(str(error))
    print("Workflow error:", error)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


All six agents executed successfully!


# Display the final result

In [29]:
print(json.dumps(state["itinerary"], indent=2, ensure_ascii=False))

{
  "source": "Hyderabad",
  "destination": "Goa",
  "start_date": "2026-11-10",
  "end_date": "2026-11-12",
  "travelers": 2,
  "selected_transport": {
    "mode": "Bus",
    "price_per_person": 1000,
    "duration_hours": 16,
    "comfort": 3,
    "total_cost": 2000,
    "within_budget": true
  },
  "selected_hotel": {
    "name": "Budget Guesthouse",
    "price_per_night": 1200,
    "rating": 3.8,
    "nights": 2,
    "total_cost": 2400,
    "total_cost_for_group": 2400,
    "cost_per_person": 1200.0
  },
  "activities": [
    {
      "name": "Relaxing Day at Calangute Beach",
      "description": "Spend a leisurely day sunbathing, swimming, and enjoying beachside shacks. Includes sunbed rental and a short sunset walk.",
      "estimated_cost_per_person": 800.0
    },
    {
      "name": "Goan Culinary Food Tour",
      "description": "Guided tasting tour of local Goan specialties such as fish curry, vindaloo, bebinca, and feni at popular eateries in Panaji.",
      "estimated_cost_

# Display the budget summary

In [30]:
budget = state["budget_summary"]

print("TRAVEL BUDGET SUMMARY")
print("-" * 35)
print(f"Total Budget:       ₹{budget['total_budget']:,.2f}")
print(f"Transport:          ₹{budget['transport_cost']:,.2f}")
print(f"Accommodation:      ₹{budget['hotel_cost']:,.2f}")
print(f"Activities:         ₹{budget['activities_cost']:,.2f}")
print(f"Food Estimate:      ₹{budget['food_estimate']:,.2f}")
print(f"Local Transport:    ₹{budget['local_transport_estimate']:,.2f}")
print("-" * 35)
print(f"Estimated Total:    ₹{budget['estimated_total']:,.2f}")
print(f"Remaining Budget:   ₹{budget['remaining_budget']:,.2f}")
print(f"Within Budget:      {budget['within_budget']}")
print(f"Review Status:      {state['review_result']['status']}")

TRAVEL BUDGET SUMMARY
-----------------------------------
Total Budget:       ₹15,000.00
Transport:          ₹2,000.00
Accommodation:      ₹2,400.00
Activities:         ₹12,400.00
Food Estimate:      ₹3,000.00
Local Transport:    ₹1,000.00
-----------------------------------
Estimated Total:    ₹20,800.00
Remaining Budget:   ₹-5,800.00
Within Budget:      False
Review Status:      NEEDS_REVIEW


In [31]:
output_path = Path("travel_itinerary.json")

with output_path.open("w", encoding="utf-8") as file:
    json.dump(
        state["itinerary"],
        file,
        indent=2,
        ensure_ascii=False
    )

print(f"Itinerary saved to: {output_path.resolve()}")

Itinerary saved to: C:\Users\Rohit singh\Desktop\ai-travel-operations-agent\notebooks\travel_itinerary.json


# LangGraph Multi-Agent Workflow

***Define the Graph State***

In [32]:
from typing import TypedDict, Any

class TravelGraphState(TypedDict, total=False):
    travel_request: dict[str, Any]
    transport_options: list[dict[str, Any]]
    hotel_options: list[dict[str, Any]]
    weather_info: dict[str, Any]
    activities: list[dict[str, Any]]
    budget_summary: dict[str, Any]
    review_result: dict[str, Any]
    itinerary: dict[str, Any]
    errors: list[str]

***Create LangGraph Nodes***

In [33]:
def transport_node(state: TravelGraphState):
    result = transport_agent(
        state["travel_request"],
        transport_data
    )
    return {"transport_options": result}


def hotel_node(state: TravelGraphState):
    result = hotel_agent(
        state["travel_request"],
        hotel_data
    )
    return {"hotel_options": result}


def weather_node(state: TravelGraphState):
    result = weather_agent(
        state["travel_request"],
        weather_data
    )
    return {"weather_info": result}


def activities_node(state: TravelGraphState):
    result = activities_agent(state["travel_request"])
    return {"activities": result}


def budget_node(state: TravelGraphState):
    result = budget_agent(
        state["travel_request"],
        state["transport_options"],
        state["hotel_options"],
        state["activities"]
    )
    return {"budget_summary": result}


def review_node(state: TravelGraphState):
    result = review_agent(
        state["travel_request"],
        state["budget_summary"],
        state["transport_options"],
        state["hotel_options"],
        state["activities"]
    )
    return {"review_result": result}

***Build the Workflow***

In [34]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(TravelGraphState)

workflow.add_node("transport", transport_node)
workflow.add_node("hotel", hotel_node)
workflow.add_node("weather", weather_node)
workflow.add_node("activities", activities_node)
workflow.add_node("budget", budget_node)
workflow.add_node("review", review_node)

workflow.add_edge(START, "transport")
workflow.add_edge("transport", "hotel")
workflow.add_edge("hotel", "weather")
workflow.add_edge("weather", "activities")
workflow.add_edge("activities", "budget")
workflow.add_edge("budget", "review")
workflow.add_edge("review", END)

travel_graph = workflow.compile()

print("LangGraph workflow compiled successfully!")

LangGraph workflow compiled successfully!


***Run the Complete Graph***

In [35]:
initial_graph_state = {
    "travel_request": travel_request,
    "errors": []
}

final_state = travel_graph.invoke(initial_graph_state)

print("Workflow completed!")
print("Review:", final_state["review_result"])
print("Budget:", final_state["budget_summary"])

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Workflow completed!
Review: {'status': 'NEEDS_REVIEW', 'issues': ['Estimated trip cost exceeds the budget.'], 'verified_live_prices': False, 'message': 'Review the issues before using this plan.'}
Budget: {'currency': 'INR', 'total_budget': 15000, 'transport_cost': 2000, 'hotel_cost': 2400, 'activities_cost': 13200.0, 'food_estimate': 3000, 'local_transport_estimate': 1000, 'estimated_total': 21600.0, 'remaining_budget': -6600.0, 'within_budget': False, 'note': 'All costs are illustrative estimates.'}


***Generate the Final Itinerary***

In [36]:
final_itinerary = {
    "source": travel_request["source"],
    "destination": travel_request["destination"],
    "start_date": travel_request["start_date"],
    "end_date": travel_request["end_date"],
    "travelers": travel_request["travelers"],
    "transport_options": final_state["transport_options"],
    "hotel_options": final_state["hotel_options"],
    "weather": final_state["weather_info"],
    "activities": final_state["activities"],
    "budget": final_state["budget_summary"],
    "review": final_state["review_result"]
}

print(json.dumps(final_itinerary, indent=2, ensure_ascii=False))

{
  "source": "Hyderabad",
  "destination": "Goa",
  "start_date": "2026-11-10",
  "end_date": "2026-11-12",
  "travelers": 2,
  "transport_options": [
    {
      "mode": "Bus",
      "price_per_person": 1000,
      "duration_hours": 16,
      "comfort": 3,
      "total_cost": 2000,
      "within_budget": true
    },
    {
      "mode": "Train",
      "price_per_person": 1200,
      "duration_hours": 14,
      "comfort": 4,
      "total_cost": 2400,
      "within_budget": true
    },
    {
      "mode": "Flight",
      "price_per_person": 4500,
      "duration_hours": 1.5,
      "comfort": 5,
      "total_cost": 9000,
      "within_budget": true
    }
  ],
  "hotel_options": [
    {
      "name": "Budget Guesthouse",
      "price_per_night": 1200,
      "rating": 3.8,
      "nights": 2,
      "total_cost": 2400,
      "total_cost_for_group": 2400,
      "cost_per_person": 1200.0
    },
    {
      "name": "Beachside Hotel",
      "price_per_night": 2500,
      "rating": 4.3,
      "ni

# MCP Integration for External Tools

In [37]:
# Create an MCP Server
from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("travel-operations-tools")

C:\Users\Rohit singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic_settings\sources\utils.py:47: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(


***Create a Transport Search Tool***

In [38]:
from datetime import date
from typing import Any


@mcp_server.tool()
def search_transport(
    source: str,
    destination: str,
    travelers: int = 1
) -> list[dict[str, Any]]:
    """
    Return demonstration transport options for a route.

    Prices and schedules are illustrative, not live data.
    """
    if travelers < 1:
        raise ValueError("Travelers must be at least 1.")

    demo_options = [
        {
            "mode": "Flight",
            "price_per_person": 4500,
            "duration_hours": 1.5
        },
        {
            "mode": "Train",
            "price_per_person": 1200,
            "duration_hours": 14
        },
        {
            "mode": "Bus",
            "price_per_person": 1000,
            "duration_hours": 16
        }
    ]

    results = []

    for option in demo_options:
        results.append({
            **option,
            "source": source,
            "destination": destination,
            "travelers": travelers,
            "estimated_total": option["price_per_person"] * travelers,
            "data_source": "DEMO"
        })

    return results

***Create a Hotel Search Tool***

In [39]:
@mcp_server.tool()
def search_hotels(
    destination: str,
    nights: int,
    travelers: int = 1
) -> list[dict[str, Any]]:
    """
    Return demonstration hotel options and estimated accommodation costs.
    """
    if nights < 1:
        raise ValueError("Nights must be at least 1.")

    if travelers < 1:
        raise ValueError("Travelers must be at least 1.")

    demo_hotels = [
        {"name": "Budget Guesthouse", "price_per_night": 1200},
        {"name": "Beachside Hotel", "price_per_night": 2500},
        {"name": "Premium Resort", "price_per_night": 5000}
    ]

    return [
        {
            **hotel,
            "destination": destination,
            "nights": nights,
            "travelers": travelers,
            "estimated_total": hotel["price_per_night"] * nights,
            "data_source": "DEMO"
        }
        for hotel in demo_hotels
    ]

***Create a Weather Tool***

In [40]:
@mcp_server.tool()
def get_weather(destination: str, travel_date: str) -> dict[str, str]:
    """
    Return a placeholder weather result, not a live forecast.
    """
    return {
        "destination": destination,
        "travel_date": travel_date,
        "forecast": "Not available from demo data",
        "data_source": "DEMO",
        "recommendation": "Check a live weather provider before departure."
    }

***Verify Tool Definitions***

In [41]:
print("MCP server initialized:", mcp_server.name)
print("Transport tool:", search_transport.__name__)
print("Hotel tool:", search_hotels.__name__)
print("Weather tool:", get_weather.__name__)

MCP server initialized: travel-operations-tools
Transport tool: search_transport
Hotel tool: search_hotels
Weather tool: get_weather


***Test the Tool Logic***

In [42]:
transport_results = search_transport(
    source="Hyderabad",
    destination="Goa",
    travelers=2
)

hotel_results = search_hotels(
    destination="Goa",
    nights=2,
    travelers=2
)

weather_result = get_weather(
    destination="Goa",
    travel_date="2026-11-10"
)

print("Transport Results:")
print(json.dumps(transport_results, indent=2))

print("\nHotel Results:")
print(json.dumps(hotel_results, indent=2))

print("\nWeather Result:")
print(json.dumps(weather_result, indent=2))

Transport Results:
[
  {
    "mode": "Flight",
    "price_per_person": 4500,
    "duration_hours": 1.5,
    "source": "Hyderabad",
    "destination": "Goa",
    "travelers": 2,
    "estimated_total": 9000,
    "data_source": "DEMO"
  },
  {
    "mode": "Train",
    "price_per_person": 1200,
    "duration_hours": 14,
    "source": "Hyderabad",
    "destination": "Goa",
    "travelers": 2,
    "estimated_total": 2400,
    "data_source": "DEMO"
  },
  {
    "mode": "Bus",
    "price_per_person": 1000,
    "duration_hours": 16,
    "source": "Hyderabad",
    "destination": "Goa",
    "travelers": 2,
    "estimated_total": 2000,
    "data_source": "DEMO"
  }
]

Hotel Results:
[
  {
    "name": "Budget Guesthouse",
    "price_per_night": 1200,
    "destination": "Goa",
    "nights": 2,
    "travelers": 2,
    "estimated_total": 2400,
    "data_source": "DEMO"
  },
  {
    "name": "Beachside Hotel",
    "price_per_night": 2500,
    "destination": "Goa",
    "nights": 2,
    "travelers": 2,
  

# MCP Client Integration

### What is MCP?

MCP (Model Context Protocol) is a standard protocol that allows AI applications to communicate with external tools and data sources.

In our AI Travel Operations Agent project, MCP connects our application to travel-related tools through an MCP client and MCP server.

### Why Do We Need MCP?

- Connect AI agents with external tools.
- Retrieve transport options.
- Retrieve hotel options.
- Get weather information.
- Reuse tools across different AI workflows.
- Separate tool implementation from the main AI application.

### MCP Architecture

1. MCP Server: Exposes travel tools.
2. MCP Client: Connects to the MCP server.
3. MCP Tools: Provide transport, hotel, and weather information.
4. AI Agents: Use tool responses to prepare travel recommendations.
5. LangGraph: Coordinates the agents and workflow.

### Tools in Our Project

| Tool | Purpose |
|---|---|
| search_transport | Retrieves transport options |
| search_hotels | Retrieves hotel options |
| get_weather | Retrieves weather information |

### Expected Outcome

By completing this step, our application will be able to discover MCP tools, call them through the MCP client, and use their responses in the AI Travel Operations Agent workflow.

### Important Note

Currently, our MCP tools return demonstration data. Live transport prices, hotel availability, and weather forecasts are not connected yet.

In [58]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from pathlib import Path
import sys

# Use the paths already configured in your project
print("Project Root:", PROJECT_ROOT)
print("MCP Server Path:", MCP_SERVER_PATH)

assert MCP_SERVER_PATH.exists(), "MCP server file not found!"

server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(MCP_SERVER_PATH)]
)

print("MCP server configuration created successfully!")

Project Root: C:\Users\Rohit singh\Desktop\ai-travel-operations-agent
MCP Server Path: C:\Users\Rohit singh\Desktop\ai-travel-operations-agent\mcp-servers\travel_tools_server.py
MCP server configuration created successfully!


***Connect and Discover Tools***

In [59]:
async def discover_mcp_tools():
    with open(MCP_LOG_PATH, "a", encoding="utf-8") as server_log:
        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):

            async with ClientSession(read, write) as session:
                await session.initialize()

                response = await session.list_tools()

                print("Available MCP Tools:\n")

                for tool in response.tools:
                    print(f"Tool Name: {tool.name}")
                    print(f"Description: {tool.description}")
                    print("-" * 40)

                return [tool.name for tool in response.tools]


available_tools = await discover_mcp_tools()

print("Discovered tools:", available_tools)

Available MCP Tools:

Tool Name: search_transport
Description: Return sample transport options for a route.
----------------------------------------
Tool Name: search_hotels
Description: Return sample hotel options and estimated costs.
----------------------------------------
Tool Name: get_weather
Description: Return a placeholder, not a live weather forecast.
----------------------------------------
Discovered tools: ['search_transport', 'search_hotels', 'get_weather']


***Create a response extraction function***

In [60]:
def extract_mcp_result(response):
    """
    Extract clean Python data from an MCP tool response.
    """

    if response.isError:
        raise RuntimeError(
            f"MCP tool call failed: {response.content}"
        )

    # Prefer structured response data
    if response.structuredContent:
        return response.structuredContent.get("result")

    # Fallback: parse JSON from text content
    results = []

    for item in response.content:
        if item.type == "text":
            results.append(json.loads(item.text))

    if len(results) == 1:
        return results[0]

    return results

***Run and display the results***

In [63]:
import json
mcp_results = await test_mcp_tools()

print("TRANSPORT OPTIONS")
print(json.dumps(mcp_results["transport"], indent=2))

print("\nHOTEL OPTIONS")
print(json.dumps(mcp_results["hotels"], indent=2))

print("\nWEATHER INFORMATION")
print(json.dumps(mcp_results["weather"], indent=2))

Transport: meta=None content=[TextContent(type='text', text='{\n  "mode": "Flight",\n  "price_per_person": 4500,\n  "duration_hours": 1.5,\n  "source": "Hyderabad",\n  "destination": "Goa",\n  "travelers": 2,\n  "estimated_total": 9000,\n  "data_source": "DEMO"\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "mode": "Train",\n  "price_per_person": 1200,\n  "duration_hours": 14,\n  "source": "Hyderabad",\n  "destination": "Goa",\n  "travelers": 2,\n  "estimated_total": 2400,\n  "data_source": "DEMO"\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "mode": "Bus",\n  "price_per_person": 1000,\n  "duration_hours": 16,\n  "source": "Hyderabad",\n  "destination": "Goa",\n  "travelers": 2,\n  "estimated_total": 2000,\n  "data_source": "DEMO"\n}', annotations=None, meta=None)] structuredContent={'result': [{'mode': 'Flight', 'price_per_person': 4500, 'duration_hours': 1.5, 'source': 'Hyderabad', 'destination': 'Goa', 'travelers': 2, 'estimated_total

TypeError: tuple indices must be integers or slices, not str

In [73]:
async def fetch_mcp_travel_data(request):
    """Fetch transport, hotel, and weather data from the MCP server."""

    nights = (
        __import__("datetime").date.fromisoformat(request["end_date"])
        - __import__("datetime").date.fromisoformat(request["start_date"])
    ).days

    if nights < 1:
        raise ValueError("end_date must be after start_date.")

    with open(MCP_LOG_PATH, "a", encoding="utf-8") as server_log:
        async with stdio_client(
            server_params,
            errlog=server_log
        ) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                transport_response = await session.call_tool(
                    "search_transport",
                    arguments={
                        "source": request["source"],
                        "destination": request["destination"],
                        "travelers": request["travelers"],
                    },
                )

                hotel_response = await session.call_tool(
                    "search_hotels",
                    arguments={
                        "destination": request["destination"],
                        "nights": nights,
                        "travelers": request["travelers"],
                    },
                )

                weather_response = await session.call_tool(
                    "get_weather",
                    arguments={
                        "destination": request["destination"],
                        "travel_date": request["start_date"],
                    },
                )

                return {
                    "transport": extract_mcp_result(transport_response),
                    "hotels": extract_mcp_result(hotel_response),
                    "weather": extract_mcp_result(weather_response),
                }

***Fetch and inspect the data***

In [74]:
mcp_travel_data = await fetch_mcp_travel_data(travel_request)

print("Transport options:", len(mcp_travel_data["transport"]))
print("Hotel options:", len(mcp_travel_data["hotels"]))

print("\nFull MCP response:")
print(json.dumps(mcp_travel_data, indent=2, ensure_ascii=False))

Transport options: 3
Hotel options: 3

Full MCP response:
{
  "transport": [
    {
      "mode": "Flight",
      "price_per_person": 4500,
      "duration_hours": 1.5,
      "source": "Hyderabad",
      "destination": "Goa",
      "travelers": 2,
      "estimated_total": 9000,
      "data_source": "DEMO"
    },
    {
      "mode": "Train",
      "price_per_person": 1200,
      "duration_hours": 14,
      "source": "Hyderabad",
      "destination": "Goa",
      "travelers": 2,
      "estimated_total": 2400,
      "data_source": "DEMO"
    },
    {
      "mode": "Bus",
      "price_per_person": 1000,
      "duration_hours": 16,
      "source": "Hyderabad",
      "destination": "Goa",
      "travelers": 2,
      "estimated_total": 2000,
      "data_source": "DEMO"
    }
  ],
  "hotels": [
    {
      "name": "Budget Guesthouse",
      "price_per_night": 1200,
      "destination": "Goa",
      "nights": 2,
      "travelers": 2,
      "estimated_total": 2400,
      "data_source": "DEMO"
   

***Transport agent***

***Create MCP-backed agent functions***

In [75]:
def mcp_transport_agent(request, mcp_data):
    options = mcp_data["transport"]

    if not options:
        return {"options": [], "recommendation": "No transport options found."}

    cheapest = min(options, key=lambda item: item["estimated_total"])
    fastest = min(options, key=lambda item: item["duration_hours"])

    return {
        "options": options,
        "cheapest_option": cheapest,
        "fastest_option": fastest,
        "travelers": request["travelers"],
        "data_source": "MCP",
    }

***Hotel agent***

In [76]:
def mcp_hotel_agent(mcp_data):
    hotels = mcp_data["hotels"]

    if not hotels:
        return {"options": [], "recommendation": "No hotel options found."}

    cheapest = min(hotels, key=lambda item: item["estimated_total"])

    return {
        "options": hotels,
        "cheapest_option": cheapest,
        "data_source": "MCP",
    }

***Weather agent***

In [77]:
def mcp_weather_agent(mcp_data):
    weather = mcp_data["weather"]

    return {
        "destination": weather.get("destination"),
        "travel_date": weather.get("travel_date"),
        "forecast": weather.get("forecast", "Forecast unavailable"),
        "data_source": weather.get("data_source", "UNKNOWN"),
    }

***Test the agents***

In [78]:
transport_result = mcp_transport_agent(
    travel_request, mcp_travel_data
)

hotel_result = mcp_hotel_agent(mcp_travel_data)
weather_result = mcp_weather_agent(mcp_travel_data)

print("TRANSPORT AGENT")
print(json.dumps(transport_result, indent=2, ensure_ascii=False))

print("\nHOTEL AGENT")
print(json.dumps(hotel_result, indent=2, ensure_ascii=False))

print("\nWEATHER AGENT")
print(json.dumps(weather_result, indent=2, ensure_ascii=False))

TRANSPORT AGENT
{
  "options": [
    {
      "mode": "Flight",
      "price_per_person": 4500,
      "duration_hours": 1.5,
      "source": "Hyderabad",
      "destination": "Goa",
      "travelers": 2,
      "estimated_total": 9000,
      "data_source": "DEMO"
    },
    {
      "mode": "Train",
      "price_per_person": 1200,
      "duration_hours": 14,
      "source": "Hyderabad",
      "destination": "Goa",
      "travelers": 2,
      "estimated_total": 2400,
      "data_source": "DEMO"
    },
    {
      "mode": "Bus",
      "price_per_person": 1000,
      "duration_hours": 16,
      "source": "Hyderabad",
      "destination": "Goa",
      "travelers": 2,
      "estimated_total": 2000,
      "data_source": "DEMO"
    }
  ],
  "cheapest_option": {
    "mode": "Bus",
    "price_per_person": 1000,
    "duration_hours": 16,
    "source": "Hyderabad",
    "destination": "Goa",
    "travelers": 2,
    "estimated_total": 2000,
    "data_source": "DEMO"
  },
  "fastest_option": {
    "mod

## Budget agent aur travel plan
***Budget agent***

In [79]:
def mcp_budget_agent(request, transport_result, hotel_result):
    budget = request["budget"]

    cheapest_transport = transport_result.get("cheapest_option")
    cheapest_hotel = hotel_result.get("cheapest_option")

    if not cheapest_transport or not cheapest_hotel:
        return {
            "budget": budget,
            "estimated_total": None,
            "remaining_budget": None,
            "status": "Cannot calculate: transport or hotel data missing.",
        }

    transport_cost = cheapest_transport["estimated_total"]
    hotel_cost = cheapest_hotel["estimated_total"]
    estimated_total = transport_cost + hotel_cost

    return {
        "budget": budget,
        "transport_cost": transport_cost,
        "hotel_cost": hotel_cost,
        "estimated_total": estimated_total,
        "remaining_budget": budget - estimated_total,
        "status": (
            "Within budget"
            if estimated_total <= budget
            else "Over budget"
        ),
        "currency": request.get("currency", "INR"),
        "note": "Excludes food, activities, local transport, and other expenses.",
    }

***Generate a travel summary using Groq***

In [80]:
def generate_travel_summary(
    request,
    transport_result,
    hotel_result,
    weather_result,
    budget_result,
):
    prompt = f"""
Create a practical travel summary using only the data below.

TRAVEL REQUEST:
{json.dumps(request, ensure_ascii=False)}

TRANSPORT OPTIONS:
{json.dumps(transport_result, ensure_ascii=False)}

HOTEL OPTIONS:
{json.dumps(hotel_result, ensure_ascii=False)}

WEATHER INFORMATION:
{json.dumps(weather_result, ensure_ascii=False)}

BUDGET ANALYSIS:
{json.dumps(budget_result, ensure_ascii=False)}

Requirements:
1. Recommend suitable transport and hotel options.
2. Explain the estimated transport and hotel costs.
3. Mention whether the available options fit the budget.
4. Clearly mention that weather is a demo placeholder if no live forecast is connected.
5. Do not invent prices, availability, or weather conditions.
6. Use clear English with headings and bullet points.
7. State that food and activity costs are excluded from the current estimate.
"""

    return ask_llm(
        prompt,
        system_prompt=(
            "You are an AI travel operations assistant. "
            "Use only the supplied data and clearly identify limitations."
        ),
        temperature=0.3,
        max_tokens=700,
    )

***Run the budget and summary agents***

In [81]:
budget_result = mcp_budget_agent(
    travel_request,
    transport_result,
    hotel_result,
)

print("BUDGET ANALYSIS")
print(json.dumps(budget_result, indent=2, ensure_ascii=False))

travel_summary = generate_travel_summary(
    travel_request,
    transport_result,
    hotel_result,
    weather_result,
    budget_result,
)

print("\nAI TRAVEL SUMMARY")
print(travel_summary)

BUDGET ANALYSIS
{
  "budget": 15000,
  "transport_cost": 2000,
  "hotel_cost": 2400,
  "estimated_total": 4400,
  "remaining_budget": 10600,
  "status": "Within budget",
  "currency": "INR",
  "note": "Excludes food, activities, local transport, and other expenses."
}


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



AI TRAVEL SUMMARY
**Travel Summary – Hyderabad ➜ Goa (10 Nov 2026 – 12 Nov 2026)**  

---

### 1. Recommended Transport  
| Option | Cost (INR) | Duration | Why it fits |
|--------|------------|----------|-------------|
| **Bus (Cheapest)** | **2,000** (₹1,000 per person) | 16 hours | Lowest price, leaves the largest amount of budget for food, activities and local transport. |
| **Flight (Fastest)** | 9,000 (₹4,500 per person) | 1.5 hours | Quickest way to reach Goa; still within the overall budget if you are willing to spend more on transport. |
| **Train** | 2,400 (₹1,200 per person) | 14 hours | Slightly more expensive than the bus but still cheap; a middle‑ground option. |

**Recommendation:**  
- **Primary choice:** **Bus** – cheapest and keeps the trip well within budget.  
- **Alternative:** **Flight** – if saving travel time is a priority and you can allocate more of the budget to transport.

---

### 2. Recommended Hotel  
| Hotel | Cost (INR) | Nights | Total |
|-------|----

# Integrate MCP with LangGraph
## Define the workflow state

In [83]:
from typing import TypedDict, Any

class MCPTravelState(TypedDict, total=False):
    travel_request: dict
    mcp_data: dict
    transport_result: dict
    hotel_result: dict
    weather_result: dict
    budget_result: dict
    final_summary: str
    errors: list

***Define LangGraph nodes***

In [84]:
def transport_node(state: MCPTravelState):
    result = mcp_transport_agent(
        state["travel_request"],
        state["mcp_data"],
    )
    return {"transport_result": result}


def hotel_node(state: MCPTravelState):
    result = mcp_hotel_agent(state["mcp_data"])
    return {"hotel_result": result}


def weather_node(state: MCPTravelState):
    result = mcp_weather_agent(state["mcp_data"])
    return {"weather_result": result}


def budget_node(state: MCPTravelState):
    result = mcp_budget_agent(
        state["travel_request"],
        state["transport_result"],
        state["hotel_result"],
    )
    return {"budget_result": result}


def summary_node(state: MCPTravelState):
    summary = generate_travel_summary(
        state["travel_request"],
        state["transport_result"],
        state["hotel_result"],
        state["weather_result"],
        state["budget_result"],
    )
    return {"final_summary": summary}

***Build the LangGraph workflow***

In [85]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(MCPTravelState)

builder.add_node("transport", transport_node)
builder.add_node("hotel", hotel_node)
builder.add_node("weather", weather_node)
builder.add_node("budget", budget_node)
builder.add_node("summary", summary_node)

builder.add_edge(START, "transport")
builder.add_edge("transport", "hotel")
builder.add_edge("hotel", "weather")
builder.add_edge("weather", "budget")
builder.add_edge("budget", "summary")
builder.add_edge("summary", END)

mcp_travel_graph = builder.compile()

print("MCP-integrated LangGraph compiled successfully!")

MCP-integrated LangGraph compiled successfully!


***Run the complete workflow***

In [86]:
initial_state = {
    "travel_request": travel_request,
    "mcp_data": mcp_travel_data,
    "errors": [],
}

final_state = mcp_travel_graph.invoke(initial_state)

print("FINAL TRAVEL PLAN\n")
print(final_state["final_summary"])

print("\nBUDGET RESULT\n")
print(json.dumps(
    final_state["budget_result"],
    indent=2,
    ensure_ascii=False,
))

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


FINAL TRAVEL PLAN

**Travel Summary – Hyderabad ➜ Goa (10 Nov 2026 – 12 Nov 2026)**  

---

### 1. Recommended Transport  
| Option | Price (INR) | Duration | Total for 2 travelers | Why it fits |
|--------|-------------|----------|------------------------|-------------|
| **Bus (Cheapest)** | 1,000 per person | 16 hrs | **2,000** | Lowest cost, leaves plenty of budget for food, activities and optional upgrades. |
| **Flight (Fastest)** | 4,500 per person | 1.5 hrs | **9,000** | Quickest way to reach Goa; still within the overall INR 15,000 budget if paired with the budget hotel. |

*Both the bus and the flight are available from Hyderabad to Goa for 2 travelers as shown in the transport data.*  

---

### 2. Recommended Hotel  
| Option | Price per night (INR) | Nights | Total for 2 travelers | Why it fits |
|--------|-----------------------|--------|------------------------|-------------|
| **Budget Guesthouse (Cheapest)** | 1,200 | 2 | **2,400** | Lowest accommodation cost, maximise